# 04a. Validation Candidate Sampling and Counterfactual Head Prediction

This notebook prepares the data used to estimate head weights from the idea:

$$
    v_{s j^+}(w) > v_{s j^-}(w),
$$

where $j^+$ is a video actually recommended in validation session $s$, $j^-$ is a sampled unrecommended video for the same user/session context, and

$$
    v_{sj}(w)=w^\top \hat p_{sj}.
$$

The vector $\hat p_{sj}$ contains the frozen response-model predictions

$$
    \hat p_{sj}
    =
    (\hat p^{complete}_{sj},\hat p^{long}_{sj},\hat p^{rewatch}_{sj},\hat p^{neg}_{sj}).
$$

The main goal is behavioral matching: choose weights so that real recommended videos receive higher scores than videos that were not recommended.


## Candidate-Set Rule

For this behavioral-cloning objective, we deliberately use the full observed panel to define user-level never-recommended videos. For each user $i$, define

$$
    R_i^{all}=\{j: j \text{ is ever recommended to user } i \text{ anywhere in the panel}\},
$$

and

$$
    N_i=\mathcal V\setminus R_i^{all}.
$$

For each validation positive $(i,s,j^+)$, the notebook samples

$$
    j^-\sim \operatorname{Uniform}(N_i),
$$

with `M_NEGATIVES_PER_POSITIVE` draws. This is intentional full-panel information use: a video recommended to the same user later is not allowed to be a negative for an earlier validation session.

By default, the negative universe is the whole observed video universe. Upload-time availability filtering is turned off so that $\mathcal V$ means all videos observed in the dataset. Turn `APPLY_UPLOAD_AVAILABILITY_RESTRICTION = True` only for a stricter historical-availability sensitivity check.


In [ ]:
from pathlib import Path
import json
import math
import time
import warnings

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
from torch import nn
import torch.nn.functional as F
from IPython.display import display

PROJECT_ROOT = Path('/Users/haozhangao/Desktop/RecSys Research')
PROCESSED = PROJECT_ROOT / 'KuaiRec 2.0' / 'data' / 'processed'
NEW_CODE = PROJECT_ROOT / 'python_code_new'
OUTPUT_DIR = NEW_CODE / 'outputs'
DOCS_DIR = NEW_CODE / 'docs'
MODEL_DIR = NEW_CODE / 'model_outputs'

INTERACTION_PATH = PROCESSED / 'processed_data.parquet'
GNN_DATA_PATH = PROCESSED / 'gnn_data.pt'

MODEL_CANDIDATES = [
    MODEL_DIR / 'edge_mlp_full.pt',
    MODEL_DIR / 'gnn_full.pt',
    MODEL_DIR / 'clean_gnn_edge_mlp_full.pt',
    MODEL_DIR / 'clean_gnn_gnn_full.pt',
]

RUN_MODE = 'full'  # choices: 'smoke', 'full'
MAX_VALIDATION_SESSIONS = 500 if RUN_MODE == 'smoke' else None
RANDOM_SEED = 20260615
M_NEGATIVES_PER_POSITIVE = 5
NEGATIVE_SET_DEFINITION = 'user_never_recommended_full_panel'  # choices: 'user_never_recommended_full_panel', 'not_previously_recommended'
APPLY_UPLOAD_AVAILABILITY_RESTRICTION = False
HISTORY_ALPHA = 0.1
PREDICTION_BATCH_SIZE = 65_536
WRITE_OUTPUTS = True
ALLOW_RANK_FEATURES_FOR_DIAGNOSTIC_ONLY = False

CANDIDATE_FEATURES_PATH = OUTPUT_DIR / 'validation_candidate_features.parquet'
CANDIDATE_PREDICTIONS_PATH = OUTPUT_DIR / 'validation_candidate_head_predictions.parquet'
PAIRS_PATH = OUTPUT_DIR / 'validation_winner_loser_pairs.parquet'
PAIR_DELTAS_PATH = OUTPUT_DIR / 'validation_pair_head_deltas.parquet'
DIAGNOSTICS_PATH = OUTPUT_DIR / 'validation_candidate_generation_diagnostics.json'
REPORT_PATH = DOCS_DIR / 'validation_candidate_generation_report.md'

FORBIDDEN_OUTCOME_INPUTS = {
    'watch_ratio', 'play_duration',
    'y_complete', 'y_long', 'y_rewatch', 'y_neg',
    'complete', 'long', 'rewatch', 'neg',
}
FORBIDDEN_RANK_FEATURES = {
    'sess_rank', 'sess_rank_over_sess_len', 'sess_rank_over_sess_len_sq',
    'exposure_order', 'position', 'rank_order',
}

if RUN_MODE not in {'smoke', 'full'}:
    raise ValueError("RUN_MODE must be 'smoke' or 'full'")
if NEGATIVE_SET_DEFINITION not in {'user_never_recommended_full_panel', 'not_previously_recommended'}:
    raise ValueError("NEGATIVE_SET_DEFINITION must be 'user_never_recommended_full_panel' or 'not_previously_recommended'")
NEGATIVE_EXCLUSION_POLICY = NEGATIVE_SET_DEFINITION  # Saved column name kept for compatibility with notebook 04b.

rng = np.random.default_rng(RANDOM_SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

print('device:', DEVICE)
print('run mode:', RUN_MODE)
print('negative set definition:', NEGATIVE_SET_DEFINITION)
print('apply upload availability restriction:', APPLY_UPLOAD_AVAILABILITY_RESTRICTION)
print('M negatives per positive:', M_NEGATIVES_PER_POSITIVE)


## 1. Load the Frozen Head Model Inputs

The candidate features must use exactly the same serving-safe feature columns and transformations as notebook 03. We load the notebook-02 graph artifact for feature metadata and for the validation message-passing graph. Validation query edges are scored with the train-only validation context graph, so validation edges are not inserted into message passing.


In [ ]:
data = torch.load(GNN_DATA_PATH, map_location='cpu', weights_only=False)
required_data_keys = [
    'edge_index', 'edge_index_val_mp', 'train_idx', 'val_idx', 'test_idx',
    'n_users', 'n_items', 'num_nodes', 'feature_metadata', 'session_table',
]
missing = [k for k in required_data_keys if k not in data]
if missing:
    raise KeyError(f'Clean full GNN artifact missing required keys: {missing}')

metadata = data['feature_metadata']
continuous_cols = list(metadata['continuous_cols'])
binary_cols = list(metadata['binary_cols'])
categorical_cols = list(metadata['categorical_cols'])
categorical_cardinalities = [int(x) for x in metadata['categorical_cardinalities']]
head_cols = ['p_complete', 'p_long', 'p_rewatch', 'p_neg']
model_input_cols = set(continuous_cols + binary_cols + categorical_cols)

forbidden_outcomes_used = sorted(model_input_cols & FORBIDDEN_OUTCOME_INPUTS)
if forbidden_outcomes_used:
    raise ValueError(f'Forbidden outcome columns appear in model inputs: {forbidden_outcomes_used}')

rank_features_used = sorted(model_input_cols & FORBIDDEN_RANK_FEATURES)
if rank_features_used and not ALLOW_RANK_FEATURES_FOR_DIAGNOSTIC_ONLY:
    raise ValueError(
        'Rank/position features appear in the frozen head predictor input list: '
        f'{rank_features_used}. Retrain the head predictor without rank features.'
    )

session_table = pd.DataFrame(data['session_table']).copy()
session_table['session_start_time'] = pd.to_datetime(session_table['session_start_time'])
session_table['session_end_time'] = pd.to_datetime(session_table['session_end_time'])

split_summary = (
    session_table.groupby('split', sort=False)
    .agg(sessions=('session_key', 'nunique'), observations=('n_interactions', 'sum'))
    .reindex(['train', 'val', 'test'])
)
display(split_summary)
print('feature dimensions:', {'continuous': len(continuous_cols), 'binary': len(binary_cols), 'categorical': len(categorical_cols)})
print('rank features used:', rank_features_used)
print('forbidden outcome inputs used:', forbidden_outcomes_used)


In [ ]:
model_path = next((p for p in MODEL_CANDIDATES if p.exists()), None)
if model_path is None:
    searched = '\n'.join(f'  - {p}' for p in MODEL_CANDIDATES)
    raise FileNotFoundError(f'No frozen local head model found. Checked:\n{searched}')

ckpt = torch.load(model_path, map_location='cpu', weights_only=False)
state_dict = ckpt['model_state_dict']


def infer_checkpoint_model_type(ckpt):
    keys = set(ckpt['model_state_dict'].keys())
    if any(k.startswith('node_emb.') for k in keys) or any(k.startswith('gcn_layers.') for k in keys):
        return 'gnn'
    if any(k.startswith('net.') for k in keys) or any(k.startswith('edge_encoder.') for k in keys):
        return 'edge_mlp'
    return ckpt.get('model_type', 'unknown')

checkpoint_label = ckpt.get('model_type', 'unknown')
model_type = infer_checkpoint_model_type(ckpt)
if checkpoint_label != model_type:
    warnings.warn(f'Checkpoint label says {checkpoint_label!r}; state_dict looks like {model_type!r}. Using inferred type.')

print('loaded frozen checkpoint:', model_path)
print('checkpoint label:', checkpoint_label)
print('actual model type:', model_type)
print('checkpoint metrics:', json.dumps(ckpt.get('metrics', {}), indent=2))


In [ ]:
def default_embedding_dim(cardinality):
    return min(32, max(2, int(round(float(cardinality) ** 0.25 * 4))))


class EdgeFeatureEncoder(nn.Module):
    def __init__(self, categorical_cardinalities, continuous_dim, binary_dim, embedding_dims=None):
        super().__init__()
        self.continuous_dim = int(continuous_dim)
        self.binary_dim = int(binary_dim)
        self.categorical_cardinalities = [int(c) for c in categorical_cardinalities]
        if embedding_dims is None:
            embedding_dims = [default_embedding_dim(c) for c in self.categorical_cardinalities]
        self.embedding_dims = [int(d) for d in embedding_dims]
        self.embeddings = nn.ModuleList([
            nn.Embedding(cardinality, dim)
            for cardinality, dim in zip(self.categorical_cardinalities, self.embedding_dims)
        ])
        self.output_dim = self.continuous_dim + self.binary_dim + sum(self.embedding_dims)

    def forward(self, x_cont, x_bin, x_cat):
        parts = []
        if self.continuous_dim:
            parts.append(x_cont.float())
        if self.binary_dim:
            parts.append(x_bin.float())
        for j, emb in enumerate(self.embeddings):
            codes = x_cat[:, j].long().clamp(min=0, max=emb.num_embeddings - 1)
            parts.append(emb(codes))
        return torch.cat(parts, dim=-1)


class EdgeMLP(nn.Module):
    def __init__(self, categorical_cardinalities, continuous_dim, binary_dim, hidden_dim=128, dropout=0.0, num_heads=4):
        super().__init__()
        self.edge_encoder = EdgeFeatureEncoder(categorical_cardinalities, continuous_dim, binary_dim)
        self.net = nn.Sequential(
            nn.Linear(self.edge_encoder.output_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_heads),
        )

    def encode_nodes(self, message_edge_index):
        return None

    def score_edges(self, node_emb, query_edge_index, x_cont, x_bin, x_cat):
        return self.net(self.edge_encoder(x_cont, x_bin, x_cat))


class SparseGCNConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.lin = nn.Linear(in_channels, out_channels)

    def forward(self, x, edge_index):
        num_nodes = x.shape[0]
        src, dst = edge_index
        self_loop = torch.arange(num_nodes, device=x.device)
        src_all = torch.cat([src, self_loop])
        dst_all = torch.cat([dst, self_loop])
        deg = torch.bincount(dst_all, minlength=num_nodes).float().clamp_min(1.0)
        out = torch.zeros_like(x)
        out.index_add_(0, dst_all, x[src_all])
        out = out / deg.unsqueeze(-1)
        return self.lin(out)


class CleanGNN(nn.Module):
    def __init__(self, num_nodes, categorical_cardinalities, continuous_dim, binary_dim, hidden_dim=64, num_heads=4, num_gcn_layers=3, dropout=0.0):
        super().__init__()
        self.node_emb = nn.Embedding(num_nodes, hidden_dim)
        self.gcn_layers = nn.ModuleList([SparseGCNConv(hidden_dim, hidden_dim) for _ in range(num_gcn_layers)])
        self.dropout = float(dropout)
        self.edge_encoder = EdgeFeatureEncoder(categorical_cardinalities, continuous_dim, binary_dim)
        input_dim = 2 * hidden_dim + self.edge_encoder.output_dim
        self.edge_mlp = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, num_heads),
        )

    def encode_nodes(self, message_edge_index):
        x = self.node_emb.weight
        for layer in self.gcn_layers:
            x = layer(x, message_edge_index)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return x

    def score_edges(self, node_emb, query_edge_index, x_cont, x_bin, x_cat):
        src, dst = query_edge_index
        edge_repr = self.edge_encoder(x_cont, x_bin, x_cat)
        full_repr = torch.cat([node_emb[src], node_emb[dst], edge_repr], dim=-1)
        return self.edge_mlp(full_repr)


def infer_edge_mlp_hidden_dim(state_dict):
    if 'net.0.weight' in state_dict:
        return int(state_dict['net.0.weight'].shape[0])
    return int(state_dict['edge_mlp.0.weight'].shape[0])

hidden_dim = int(ckpt.get('hidden_dim', 64))
dropout = float(ckpt.get('dropout', 0.0))

if model_type == 'edge_mlp':
    hidden_dim = infer_edge_mlp_hidden_dim(state_dict)
    head_model = EdgeMLP(categorical_cardinalities, len(continuous_cols), len(binary_cols), hidden_dim=hidden_dim, dropout=dropout, num_heads=4)
elif model_type == 'gnn':
    if 'node_emb.weight' in state_dict:
        hidden_dim = int(state_dict['node_emb.weight'].shape[1])
    head_model = CleanGNN(int(data['num_nodes']), categorical_cardinalities, len(continuous_cols), len(binary_cols), hidden_dim=hidden_dim, dropout=dropout, num_heads=4)
else:
    raise ValueError(f'Unknown model_type: {model_type}')

head_model.load_state_dict(state_dict)
head_model.to(DEVICE)
head_model.eval()
print('head model loaded with hidden_dim:', hidden_dim)


## 2. Load the Interaction Table and Select Validation Sessions

Positive examples are unique session-video pairs from validation sessions. If the same video appears more than once inside a session, it contributes one positive candidate. This matches the recommendation-level question: whether a video was selected into the session slate.


In [ ]:
parquet_cols = set(pq.read_schema(INTERACTION_PATH).names)
item_static_cols = [
    'i_aspect_ratio', 'i_video_duration', 'i_author_id', 'i_video_type', 'i_upload_type',
    'i_visible_status', 'i_music_id', 'i_video_tag_id', 'i_video_tag_name',
    'i_cat_level1_id', 'i_cat_level1_name', 'i_cat_level2_id', 'i_cat_level2_name',
    'i_cat_level3_id', 'i_cat_level3_name',
]
base_cols = [
    'user_id', 'video_id', 'time', 'date', 'timestamp', 'burst_id', 'session', 'sess_rank', 'sess_len',
    'y_complete', 'y_long', 'y_rewatch', 'y_neg', 'watch_ratio',
]
needed_cols = sorted((set(base_cols) | model_input_cols | set(item_static_cols) | {'i_age_since_upload_days'}) & parquet_cols)
missing_input_cols = sorted(model_input_cols - set(needed_cols))
if missing_input_cols:
    raise KeyError(f'Model input columns missing from processed_data: {missing_input_cols}')

df = pd.read_parquet(INTERACTION_PATH, columns=needed_cols).copy()
df['source_row_id'] = np.arange(len(df), dtype=np.int64)
df['time'] = pd.to_datetime(df['time'])
df['session_key'] = df['user_id'].astype(str) + '||' + df['session'].astype(str)

session_lookup = session_table.set_index('session_key')
df['split'] = df['session_key'].map(session_lookup['split'])
df['session_start_time'] = df['session_key'].map(session_lookup['session_start_time'])
if df['split'].isna().any():
    raise ValueError('Some interactions did not receive a clean split label')

val_session_table = session_table.loc[session_table['split'].eq('val')].copy()
if MAX_VALIDATION_SESSIONS is not None:
    sampled_keys = rng.choice(
        val_session_table['session_key'].to_numpy(),
        size=min(MAX_VALIDATION_SESSIONS, len(val_session_table)),
        replace=False,
    )
    active_val_keys = set(sampled_keys.tolist())
else:
    active_val_keys = set(val_session_table['session_key'].tolist())

df_val = df.loc[df['session_key'].isin(active_val_keys)].copy()
if not df_val['split'].eq('val').all():
    raise ValueError('Non-validation rows appeared in df_val')

val_idx_set = set(map(int, data['val_idx'].cpu().numpy().tolist()))
row_id_set = set(map(int, df_val['source_row_id'].to_numpy().tolist()))
if MAX_VALIDATION_SESSIONS is None and row_id_set != val_idx_set:
    raise ValueError('Full validation rows do not match gnn_data val_idx')

df_val = df_val.sort_values(['session_key', 'sess_rank', 'time', 'video_id'], kind='mergesort').reset_index(drop=True)
df_val['rank_order'] = df_val.groupby('session_key', observed=True).cumcount() + 1

positive_session_videos = (
    df_val.sort_values(['session_key', 'rank_order', 'video_id'], kind='mergesort')
    .drop_duplicates(['session_key', 'video_id'], keep='first')
    [['session_key', 'user_id', 'session', 'burst_id', 'session_start_time', 'video_id', 'rank_order']]
    .copy()
)
positive_session_videos['positive_rank_order'] = positive_session_videos['rank_order'].astype(np.int32)
positive_session_videos = positive_session_videos.drop(columns=['rank_order'])

candidate_recomputed_cols = {
    'i_age_since_upload_days',
    'hist_cat_ema_complete',
    'hist_author_recency_days',
    'hist_last_complete_author',
    'hist_has_author_history',
}
item_model_cols = {col for col in model_input_cols if col.startswith('i_')}
session_model_cols = model_input_cols - item_model_cols - candidate_recomputed_cols - {'video_id'}
session_first_cols = sorted(session_model_cols | {'session_key', 'user_id', 'session', 'burst_id', 'session_start_time', 'split'})
session_context = (
    df_val.sort_values(['session_key', 'rank_order'], kind='mergesort')
    .drop_duplicates('session_key', keep='first')
    [session_first_cols]
    .copy()
)

summary = {
    'run_mode': RUN_MODE,
    'validation_rows': int(len(df_val)),
    'validation_sessions': int(df_val['session_key'].nunique()),
    'validation_users': int(df_val['user_id'].nunique()),
    'positive_session_videos': int(len(positive_session_videos)),
}
print(json.dumps(summary, indent=2))
display(positive_session_videos.head())


## 3. Build Pre-Session State and Video Metadata

For every target validation session, candidate features are evaluated at the start of the session. Therefore candidate-specific history variables are based on the user's state before the session starts, not on outcomes later in the same session.

Recomputed candidate-specific features:

| Feature | Candidate dependence | Construction |
|---|---|---|
| `i_age_since_upload_days` | video | target session start minus reconstructed upload date |
| `hist_cat_ema_complete` | user, session, video category | pre-session user/category completion EMA |
| `hist_author_recency_days` | user, session, video author | days since prior same-author exposure in the same burst |
| `hist_last_complete_author` | user, session, video author | previous completion outcome for same author in the same burst |
| `hist_has_author_history` | user, session, video author | indicator for same-author history in the same burst |


In [ ]:
def build_video_table(frame):
    available_item_cols = [c for c in item_static_cols if c in frame.columns]
    age_col = 'i_age_since_upload_days'
    work_cols = ['video_id', 'time'] + available_item_cols
    if age_col in frame.columns:
        work_cols.append(age_col)
    work = frame[work_cols].copy()
    if age_col in work.columns:
        age_days = pd.to_numeric(work[age_col], errors='coerce')
        work['i_upload_dt_est'] = work['time'] - pd.to_timedelta(age_days, unit='D')
    else:
        work['i_upload_dt_est'] = pd.NaT
    work = work.sort_values(['video_id', 'time'], kind='mergesort')
    per_video = work.drop_duplicates('video_id', keep='first').copy()
    keep = ['video_id'] + available_item_cols + ['i_upload_dt_est']
    per_video = per_video[keep]
    per_video = per_video.loc[per_video['video_id'].between(0, int(data['n_items']) - 1)].copy()
    return per_video

video_table = build_video_table(df)
all_video_ids = np.sort(video_table['video_id'].astype(np.int64).unique())
video_upload_times = (
    video_table.set_index('video_id').reindex(all_video_ids)['i_upload_dt_est']
    .to_numpy(dtype='datetime64[ns]')
)
print('candidate video universe:', len(all_video_ids))
display(video_table.head())


In [ ]:
def build_user_all_seen(frame, target_users):
    out = {}
    sub = frame.loc[frame['user_id'].isin(target_users), ['user_id', 'video_id']]
    for uid, vids in sub.groupby('user_id', sort=False)['video_id']:
        out[int(uid)] = set(map(int, pd.unique(vids)))
    return out


def build_pre_session_states(frame, target_session_keys):
    target_session_keys = set(target_session_keys)
    target_users = set(session_context.loc[session_context['session_key'].isin(target_session_keys), 'user_id'].astype(int))
    states = {}
    t0 = time.time()

    sorted_frame = (
        frame.loc[frame['user_id'].isin(target_users)]
        .sort_values(['user_id', 'time', 'video_id'], kind='mergesort')
    )

    for user_num, (uid, user_df) in enumerate(sorted_frame.groupby('user_id', observed=True, sort=False), start=1):
        seen_videos = set()
        cat_ema = {}
        author_state = {}

        for session_key, session_rows in user_df.groupby('session_key', observed=True, sort=False):
            if session_key in target_session_keys:
                states[session_key] = {
                    'seen_videos_before': set(seen_videos),
                    'cat_ema_before': dict(cat_ema),
                    'author_state_before': dict(author_state),
                }

            for row in session_rows.itertuples(index=False):
                vid = int(row.video_id)
                seen_videos.add(vid)

                cat = getattr(row, 'i_cat_level1_id', np.nan)
                if pd.notna(cat):
                    cat_key = int(cat)
                    y = float(getattr(row, 'y_complete'))
                    old = cat_ema.get(cat_key)
                    cat_ema[cat_key] = y if old is None else HISTORY_ALPHA * y + (1.0 - HISTORY_ALPHA) * old

                author = getattr(row, 'i_author_id', np.nan)
                burst = getattr(row, 'burst_id', np.nan)
                if pd.notna(author) and pd.notna(burst):
                    author_key = (int(burst), str(author))
                    author_state[author_key] = (pd.Timestamp(row.time), float(getattr(row, 'y_complete')))

        if user_num % 500 == 0:
            print(f'processed {user_num:,} users; states={len(states):,}; elapsed={time.time() - t0:.1f}s', flush=True)

    missing_states = target_session_keys - set(states)
    if missing_states:
        raise ValueError(f'Missing pre-session states for {len(missing_states)} target sessions')
    return states

active_val_keys_list = sorted(active_val_keys)
target_users = set(session_context['user_id'].astype(int))
user_all_seen = build_user_all_seen(df, target_users)
pre_session_states = build_pre_session_states(df, active_val_keys_list)
print('pre-session states:', len(pre_session_states))
print('user all-seen maps:', len(user_all_seen))


## 4. Sample Unrecommended Videos

For each positive session-video pair, sample `M_NEGATIVES_PER_POSITIVE` negatives from the eligible pool. The default pool excludes every video ever observed for the same user, so the sampled negatives are true user-level never-observed videos in the available panel.


In [ ]:
def available_video_ids_for_session(session_start_time):
    if not APPLY_UPLOAD_AVAILABILITY_RESTRICTION:
        return all_video_ids
    t = np.datetime64(pd.Timestamp(session_start_time).to_datetime64())
    available = np.isnat(video_upload_times) | (video_upload_times <= t)
    return all_video_ids[available]


def sample_negative_pairs(positives, m_negatives):
    rows = []
    shortfall_sessions = 0
    future_seen_false_negative_count = 0
    old_policy_false_negative_count = 0
    t0 = time.time()

    grouped = positives.groupby('session_key', observed=True, sort=False)
    n_sessions = grouped.ngroups
    for session_num, (session_key, group) in enumerate(grouped, start=1):
        uid = int(group['user_id'].iloc[0])
        session = int(group['session'].iloc[0])
        burst_id = int(group['burst_id'].iloc[0])
        session_start = pd.Timestamp(group['session_start_time'].iloc[0])
        positive_videos = np.array(sorted(pd.unique(group['video_id'].astype(np.int64))), dtype=np.int64)
        current_positive_set = set(map(int, positive_videos))

        state = pre_session_states[session_key]
        prior_seen = set(map(int, state['seen_videos_before']))
        ever_seen = set(map(int, user_all_seen.get(uid, set())))
        future_seen = ever_seen - prior_seen - current_positive_set

        if NEGATIVE_SET_DEFINITION == 'user_never_recommended_full_panel':
            excluded = ever_seen | current_positive_set
            negative_type = 'uniform_user_never_recommended_full_panel'
        else:
            excluded = prior_seen | current_positive_set
            negative_type = 'uniform_not_previously_recommended'

        available_ids = available_video_ids_for_session(session_start)
        if excluded:
            eligible = available_ids[~np.isin(available_ids, np.fromiter(excluded, dtype=np.int64), assume_unique=False)]
        else:
            eligible = available_ids
        eligible_pool_size = int(len(eligible))
        if eligible_pool_size == 0:
            shortfall_sessions += 1
            continue

        if eligible_pool_size < m_negatives:
            shortfall_sessions += 1

        for pos_row in group.itertuples(index=False):
            replace = eligible_pool_size < m_negatives
            sampled = rng.choice(eligible, size=m_negatives, replace=replace)
            for draw_id, neg_vid in enumerate(sampled):
                neg_vid = int(neg_vid)
                is_future_seen = neg_vid in future_seen
                is_ever_seen = neg_vid in ever_seen
                if is_future_seen:
                    future_seen_false_negative_count += 1
                if NEGATIVE_SET_DEFINITION == 'not_previously_recommended' and is_ever_seen:
                    old_policy_false_negative_count += 1
                rows.append({
                    'pair_id': len(rows),
                    'session_key': session_key,
                    'user_id': uid,
                    'session': session,
                    'burst_id': burst_id,
                    'session_start_time': session_start,
                    'j_plus': int(pos_row.video_id),
                    'j_minus': neg_vid,
                    'positive_rank_order': int(pos_row.positive_rank_order),
                    'negative_draw_id': int(draw_id),
                    'num_negatives_sampled': int(m_negatives),
                    'eligible_pool_size': eligible_pool_size,
                    'negative_type': negative_type,
                    'negative_exclusion_policy': NEGATIVE_SET_DEFINITION,
                    'negative_prior_seen_by_user': int(neg_vid in prior_seen),
                    'negative_future_seen_by_user': int(is_future_seen),
                    'negative_ever_seen_by_user': int(is_ever_seen),
                    'upload_availability_restricted': bool(APPLY_UPLOAD_AVAILABILITY_RESTRICTION),
                    'split': 'val',
                })

        if session_num % 1000 == 0:
            print(f'sampled {session_num:,}/{n_sessions:,} sessions; pairs={len(rows):,}; elapsed={time.time() - t0:.1f}s', flush=True)

    pairs = pd.DataFrame(rows)
    diagnostics = {
        'validation_sessions_used': int(positives['session_key'].nunique()),
        'positive_videos': int(len(positives)),
        'pairs': int(len(pairs)),
        'average_negatives_per_positive': float(len(pairs) / max(len(positives), 1)),
        'negative_shortfall_sessions': int(shortfall_sessions),
        'future_seen_false_negative_count': int(future_seen_false_negative_count),
        'old_policy_ever_seen_negative_count': int(old_policy_false_negative_count),
    }
    return pairs, diagnostics

pairs, pair_sampling_diagnostics = sample_negative_pairs(positive_session_videos, M_NEGATIVES_PER_POSITIVE)
print(json.dumps(pair_sampling_diagnostics, indent=2))
display(pairs.head())


## 5. Construct Candidate Features

The candidate feature table contains one row per unique `(session_key, video_id, candidate_role)` candidate. Positives and negatives are both evaluated with the same pre-session user state and the same item metadata pipeline.


In [ ]:
def make_candidate_index(positives, pairs):
    positive_candidates = positives[[
        'session_key', 'user_id', 'session', 'burst_id', 'session_start_time', 'video_id'
    ]].drop_duplicates(['session_key', 'video_id']).copy()
    positive_candidates['candidate_role'] = 'positive'
    positive_candidates['is_observed_positive'] = 1
    positive_candidates['negative_type'] = 'observed_positive'

    negative_candidates = pairs[[
        'session_key', 'user_id', 'session', 'burst_id', 'session_start_time', 'j_minus', 'negative_type'
    ]].rename(columns={'j_minus': 'video_id'}).drop_duplicates(['session_key', 'video_id']).copy()
    negative_candidates['candidate_role'] = 'negative'
    negative_candidates['is_observed_positive'] = 0

    out = pd.concat([positive_candidates, negative_candidates], ignore_index=True)
    out = out.sort_values(['session_key', 'candidate_role', 'video_id'], kind='mergesort').reset_index(drop=True)
    return out

candidate_index = make_candidate_index(positive_session_videos, pairs)

# Session context carries user, time, and pre-session general history features.
context_cols = [
    c for c in session_context.columns
    if c not in {'session_key', 'user_id', 'session', 'burst_id', 'session_start_time', 'split'}
]
session_context_for_merge = session_context[['session_key'] + context_cols].copy()
if session_context_for_merge.columns.duplicated().any():
    duplicated = session_context_for_merge.columns[session_context_for_merge.columns.duplicated()].tolist()
    raise ValueError(f'Duplicate columns in session_context_for_merge: {duplicated}')

candidate_features = candidate_index.merge(
    session_context_for_merge,
    on='session_key',
    how='left',
    validate='many_to_one',
)

video_static_cols = [c for c in video_table.columns if c not in {'i_age_since_upload_days'}]
candidate_features = candidate_features.merge(
    video_table[video_static_cols],
    on='video_id',
    how='left',
    validate='many_to_one',
)

# Recompute item age at target session start.
if 'i_upload_dt_est' in candidate_features.columns:
    candidate_features['i_age_since_upload_days'] = (
        (pd.to_datetime(candidate_features['session_start_time']) - pd.to_datetime(candidate_features['i_upload_dt_est']))
        .dt.total_seconds() / (24 * 3600)
    ).astype('float32')
else:
    candidate_features['i_age_since_upload_days'] = np.nan

# Recompute category-specific pre-session completion EMA by merging sparse state rows.
cat_state_rows = []
for session_key, state in pre_session_states.items():
    for cat, value in state['cat_ema_before'].items():
        cat_state_rows.append((session_key, int(cat), float(value)))
cat_state = pd.DataFrame(cat_state_rows, columns=['session_key', '_cat_level1_key', '_hist_cat_ema_complete'])
candidate_features['_cat_level1_key'] = pd.to_numeric(candidate_features['i_cat_level1_id'], errors='coerce').fillna(-1).astype(np.int64)
candidate_features = candidate_features.merge(cat_state, on=['session_key', '_cat_level1_key'], how='left')
candidate_features['hist_cat_ema_complete'] = candidate_features['_hist_cat_ema_complete'].astype('float32')
candidate_features = candidate_features.drop(columns=['_cat_level1_key', '_hist_cat_ema_complete'])

# Recompute author-specific pre-session history on unique session-author keys, then merge back.
author_keys = candidate_features[[
    'session_key', 'burst_id', 'session_start_time', 'i_author_id'
]].drop_duplicates().copy()
recency = []
last_complete = []
has_history = []
for row in author_keys.itertuples(index=False):
    value = None
    if pd.notna(row.i_author_id) and pd.notna(row.burst_id):
        value = pre_session_states[row.session_key]['author_state_before'].get((int(row.burst_id), str(row.i_author_id)))
    if value is None:
        recency.append(np.nan)
        last_complete.append(np.nan)
        has_history.append(0)
    else:
        last_time, last_y = value
        recency.append((pd.Timestamp(row.session_start_time) - pd.Timestamp(last_time)).total_seconds() / (24 * 3600))
        last_complete.append(float(last_y))
        has_history.append(1)
author_keys['hist_author_recency_days'] = np.asarray(recency, dtype='float32')
author_keys['hist_last_complete_author'] = np.asarray(last_complete, dtype='float32')
author_keys['hist_has_author_history'] = np.asarray(has_history, dtype='int8')

candidate_features = candidate_features.merge(
    author_keys,
    on=['session_key', 'burst_id', 'session_start_time', 'i_author_id'],
    how='left',
    validate='many_to_one',
    suffixes=('', '_recomputed'),
)

# Keep the recomputed columns if merge suffixes appeared.
for col in ['hist_author_recency_days', 'hist_last_complete_author', 'hist_has_author_history']:
    rec_col = f'{col}_recomputed'
    if rec_col in candidate_features.columns:
        candidate_features[col] = candidate_features[rec_col]
        candidate_features = candidate_features.drop(columns=[rec_col])

missing_feature_cols = sorted(model_input_cols - set(candidate_features.columns))
if missing_feature_cols:
    raise KeyError(f'Candidate features are missing model input columns: {missing_feature_cols}')

forbidden_columns_in_candidate_features = sorted(set(candidate_features.columns) & FORBIDDEN_OUTCOME_INPUTS)
feature_diagnostics = {
    'candidate_rows': int(len(candidate_features)),
    'positive_candidates': int(candidate_features['is_observed_positive'].sum()),
    'negative_candidates': int((candidate_features['is_observed_positive'] == 0).sum()),
    'unique_candidate_session_videos': int(candidate_features[['session_key', 'video_id']].drop_duplicates().shape[0]),
    'missing_required_feature_cells': int(candidate_features[continuous_cols + binary_cols + categorical_cols].isna().sum().sum()),
    'forbidden_columns_in_candidate_features': forbidden_columns_in_candidate_features,
}
print(json.dumps(feature_diagnostics, indent=2))
display(candidate_features.head())


## 6. Predict the Four Heads for All Candidate Rows

For every candidate row, the frozen response model predicts the four heads under the candidate's session context and item metadata. These predictions are the inputs to the weight estimator in notebook 04b.


In [ ]:
def transform_continuous(frame):
    out = np.zeros((len(frame), len(continuous_cols)), dtype=np.float32)
    scaler = metadata['continuous_scaler']
    for j, col in enumerate(continuous_cols):
        params = scaler[col]
        raw = pd.to_numeric(frame[col], errors='coerce')
        mean = float(params['mean'])
        std = float(params['std'])
        if not np.isfinite(std) or std <= 1e-8:
            std = 1.0
        out[:, j] = ((raw.fillna(mean).astype(float) - mean) / (std + 1e-8)).astype(np.float32)
    return torch.from_numpy(out)


def transform_binary(frame):
    out = np.zeros((len(frame), len(binary_cols)), dtype=np.float32)
    for j, col in enumerate(binary_cols):
        raw = pd.to_numeric(frame[col], errors='coerce').fillna(0).clip(0, 1)
        out[:, j] = raw.astype(np.float32)
    return torch.from_numpy(out)


def stable_hash_codes(values, n_buckets):
    hashed = pd.util.hash_pandas_object(values.astype('string').fillna('__MISSING__'), index=False)
    return (hashed.to_numpy(dtype='uint64') % np.uint64(n_buckets)).astype('int64') + 1


def transform_categorical(frame):
    out = np.zeros((len(frame), len(categorical_cols)), dtype=np.int64)
    mappings = metadata['categorical_mappings']
    for j, col in enumerate(categorical_cols):
        info = mappings[col]
        values = frame[col].astype('string').fillna('__MISSING__').replace({'': '__MISSING__'})
        mapping = info.get('mapping')
        cardinality = int(info['cardinality'])
        if mapping is None:
            codes = stable_hash_codes(values, cardinality - 1)
        else:
            codes = values.map(mapping).fillna(0).astype(np.int64).to_numpy()
        out[:, j] = np.clip(codes, 0, cardinality - 1)
    return torch.from_numpy(out)


def compute_train_degrees(data):
    edge_index_train = data.get('edge_index_train_context', data['edge_index_train_mp'])
    flat = edge_index_train.reshape(-1)
    degree = torch.bincount(flat, minlength=int(data['num_nodes'])).cpu().numpy()
    return degree

train_degrees = compute_train_degrees(data)
print('transform functions ready')


In [ ]:
@torch.no_grad()
def predict_candidate_heads(frame):
    head_model.eval()
    outputs = []
    n = len(frame)

    if frame['video_id'].max() >= int(data['n_items']) or frame['video_id'].min() < 0:
        raise ValueError('Candidate video_id is outside the GNN item node range')

    node_emb = None
    if model_type == 'gnn':
        node_emb = head_model.encode_nodes(data['edge_index_val_mp'].to(DEVICE))

    for start in range(0, n, PREDICTION_BATCH_SIZE):
        batch = frame.iloc[start:start + PREDICTION_BATCH_SIZE]
        x_cont = transform_continuous(batch).to(DEVICE)
        x_bin = transform_binary(batch).to(DEVICE)
        x_cat = transform_categorical(batch).to(DEVICE)

        src = torch.from_numpy(batch['user_id'].to_numpy(dtype=np.int64)).long()
        dst = torch.from_numpy((int(data['n_users']) + batch['video_id'].to_numpy(dtype=np.int64))).long()
        query_edge_index = torch.stack([src, dst], dim=0).to(DEVICE)

        logits = head_model.score_edges(node_emb, query_edge_index, x_cont, x_bin, x_cat)
        outputs.append(torch.sigmoid(logits).detach().cpu().numpy())

        if start and start % (PREDICTION_BATCH_SIZE * 20) == 0:
            print(f'predicted {start:,}/{n:,} candidate rows', flush=True)

    probs = np.vstack(outputs)
    pred_cols = [
        'session_key', 'user_id', 'session', 'burst_id', 'session_start_time',
        'video_id', 'candidate_role', 'is_observed_positive', 'negative_type',
        'i_cat_level1_id', 'i_cat_level2_id', 'i_cat_level3_id', 'i_author_id',
    ]
    pred = frame[pred_cols].copy()
    pred['split'] = 'val'
    pred['p_complete'] = probs[:, 0]
    pred['p_long'] = probs[:, 1]
    pred['p_rewatch'] = probs[:, 2]
    pred['p_neg'] = probs[:, 3]

    user_nodes = pred['user_id'].to_numpy(dtype=np.int64)
    video_nodes = int(data['n_users']) + pred['video_id'].to_numpy(dtype=np.int64)
    pred['user_degree_in_train_graph'] = train_degrees[user_nodes]
    pred['video_degree_in_train_graph'] = train_degrees[video_nodes]
    pred['user_is_cold_start'] = pred['user_degree_in_train_graph'].eq(0).astype('int8')
    pred['video_is_cold_start'] = pred['video_degree_in_train_graph'].eq(0).astype('int8')
    return pred

candidate_predictions = predict_candidate_heads(candidate_features)
if candidate_predictions[head_cols].isna().any().any():
    raise ValueError('Missing candidate head predictions')
if not candidate_predictions[head_cols].apply(lambda s: s.between(0, 1).all()).all():
    raise ValueError('Some predicted heads are outside [0, 1]')

prediction_diagnostics = {
    'candidate_predictions': int(len(candidate_predictions)),
    'all_heads_in_unit_interval': True,
    'missing_head_predictions': int(candidate_predictions[head_cols].isna().sum().sum()),
    'user_cold_start_share': float(candidate_predictions['user_is_cold_start'].mean()),
    'video_cold_start_share': float(candidate_predictions['video_is_cold_start'].mean()),
}
print(json.dumps(prediction_diagnostics, indent=2))
display(candidate_predictions.head())


## 7. Build Pairwise Head Deltas and Save Artifacts

For every sampled pair, notebook 04b only needs

$$
    \Delta \hat p_i
    =
    \hat p_{s_i j_i^+} - \hat p_{s_i j_i^-}.
$$

The saved pair-delta table keeps the pair metadata and the four head deltas.


In [ ]:
plus = candidate_predictions.loc[candidate_predictions['candidate_role'].eq('positive'), ['session_key', 'video_id'] + head_cols].copy()
minus = candidate_predictions.loc[candidate_predictions['candidate_role'].eq('negative'), ['session_key', 'video_id'] + head_cols].copy()
plus = plus.rename(columns={'video_id': 'j_plus', **{c: f'{c}_plus' for c in head_cols}})
minus = minus.rename(columns={'video_id': 'j_minus', **{c: f'{c}_minus' for c in head_cols}})

pair_deltas = pairs.merge(plus, on=['session_key', 'j_plus'], how='left', validate='many_to_one')
pair_deltas = pair_deltas.merge(minus, on=['session_key', 'j_minus'], how='left', validate='many_to_one')

for raw in ['complete', 'long', 'rewatch', 'neg']:
    pair_deltas[f'delta_{raw}'] = pair_deltas[f'p_{raw}_plus'] - pair_deltas[f'p_{raw}_minus']

delta_cols = ['delta_complete', 'delta_long', 'delta_rewatch', 'delta_neg']
pair_delta_diagnostics = {
    'pair_head_delta_rows': int(len(pair_deltas)),
    'pairs_with_plus_minus_predictions': int(pair_deltas[[f'{c}_plus' for c in head_cols] + [f'{c}_minus' for c in head_cols]].notna().all(axis=1).sum()),
    'delta_missing_cells': int(pair_deltas[delta_cols].isna().sum().sum()),
    'negative_prior_seen_by_user_sum': int(pair_deltas['negative_prior_seen_by_user'].sum()),
    'negative_future_seen_by_user_sum': int(pair_deltas['negative_future_seen_by_user'].sum()),
    'negative_ever_seen_by_user_sum': int(pair_deltas['negative_ever_seen_by_user'].sum()),
}
if pair_delta_diagnostics['delta_missing_cells']:
    raise ValueError(f'Missing pair deltas: {pair_delta_diagnostics}')

report_payload = {
    'run_mode': RUN_MODE,
    'random_seed': RANDOM_SEED,
    'm_negatives': M_NEGATIVES_PER_POSITIVE,
    'negative_exclusion_policy': NEGATIVE_SET_DEFINITION,
    'upload_availability_restricted': APPLY_UPLOAD_AVAILABILITY_RESTRICTION,
    'model_path': str(model_path),
    'model_type': model_type,
    'split_summary': split_summary.reset_index().to_dict(orient='records'),
    'candidate_sample_summary': summary,
    'pair_sampling_diagnostics': pair_sampling_diagnostics,
    'feature_diagnostics': feature_diagnostics,
    'prediction_diagnostics': prediction_diagnostics,
    'pair_delta_diagnostics': pair_delta_diagnostics,
    'rank_features_used': rank_features_used,
    'forbidden_outcome_inputs_used': forbidden_outcomes_used,
}

report = f"""# Validation Candidate Generation and Head Prediction

## Data

- Run mode: {RUN_MODE}
- Full validation sessions: {int(split_summary.loc['val', 'sessions']):,}
- Full validation observations: {int(split_summary.loc['val', 'observations']):,}
- Active validation sessions used in this run: {summary['validation_sessions']:,}
- Positive session-videos: {summary['positive_session_videos']:,}

Only validation sessions are target sessions. Test sessions are not used as positive target sessions.

## Negative Sampling

For each validation positive, the notebook sampled up to `M={M_NEGATIVES_PER_POSITIVE}` negatives.

- negative set definition: `{NEGATIVE_SET_DEFINITION}`
- upload-time availability restriction: `{APPLY_UPLOAD_AVAILABILITY_RESTRICTION}`

Under the default `user_never_recommended_full_panel` definition, a sampled negative must be outside the user's full-panel recommended set. Therefore later same-user recommendations cannot be sampled as negatives. This intentionally uses full-panel recommendation information for behavioral cloning.

## Feature Construction

Copied from pre-session/session state:

- user static features
- session/context features
- general user-history features

Merged from candidate video metadata:

- item static features
- item category hierarchy
- author and tag metadata

Recomputed for each candidate:

- `i_age_since_upload_days`
- `hist_cat_ema_complete`
- `hist_author_recency_days`
- `hist_last_complete_author`
- `hist_has_author_history`

Forbidden realized outcomes were not used as model input features.

## Diagnostics

```json
{json.dumps(report_payload, indent=2, default=str)}
```
"""

if WRITE_OUTPUTS:
    candidate_features.to_parquet(CANDIDATE_FEATURES_PATH, index=False)
    candidate_predictions.to_parquet(CANDIDATE_PREDICTIONS_PATH, index=False)
    pairs.to_parquet(PAIRS_PATH, index=False)
    pair_deltas.to_parquet(PAIR_DELTAS_PATH, index=False)
    DIAGNOSTICS_PATH.write_text(json.dumps(report_payload, indent=2, default=str) + '\n')
    REPORT_PATH.write_text(report)
    print('saved candidate features:', CANDIDATE_FEATURES_PATH)
    print('saved candidate predictions:', CANDIDATE_PREDICTIONS_PATH)
    print('saved pairs:', PAIRS_PATH)
    print('saved pair deltas:', PAIR_DELTAS_PATH)
    print('saved diagnostics:', DIAGNOSTICS_PATH)
    print('saved report:', REPORT_PATH)
else:
    print('WRITE_OUTPUTS is False; skipped saving.')

print(json.dumps(pair_delta_diagnostics, indent=2))
